# 05. Графики, summary-таблицы и финальные проверки

Ноутбук строит финальные графики для отчёта, сохраняет summary-таблицы и выполняет контрольные проверки итоговых данных.

## 1. Импорт библиотек и настройка headless-режима

In [11]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TEMP_DIR = PROJECT_ROOT / ".tmp"
MPLCONFIGDIR = TEMP_DIR / "matplotlib"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("XDG_CACHE_HOME", str(TEMP_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

## 2. Пути проекта и выходные директории

In [12]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUMMARY_DIR   = PROJECT_ROOT / "report" / "tables"
FIGURES_DIR   = PROJECT_ROOT / "report" / "figures"

LISTENINGS_DATALENS_PATH       = PROCESSED_DIR / "listenings_datalens.csv"
USER_WEEKDAY_HOUR_PROFILE_PATH = PROCESSED_DIR / "user_weekday_hour_profile.csv"
LISTENING_SESSIONS_PATH        = PROCESSED_DIR / "listening_sessions.csv"
USER_FEATURES_PATH             = PROCESSED_DIR / "user_features.csv"
USER_ACTIVITY_TYPES_PATH       = PROCESSED_DIR / "user_activity_types.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_DIR, FIGURES_DIR

(PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/report/tables'),
 PosixPath('/Users/zarmeux/hse/course work 1/course-project-listener-classification/report/figures'))

## 3. Константы для порядка категорий и контрольных проверок

In [13]:
EXPECTED_USER_COUNT = 11
SHARE_TOLERANCE = 1e-5

WEEKDAY_ORDER = [
    "Понедельник",
    "Вторник",
    "Среда",
    "Четверг",
    "Пятница",
    "Суббота",
    "Воскресенье",
]

DAY_PART_ORDER = [
    "Ночь",
    "Утро",
    "День",
    "Вечер",
]

## 4. Служебные функции загрузки данных и построения подписей

In [14]:
def load_data() -> dict[str, pd.DataFrame]:
    listenings_df = pd.read_csv(LISTENINGS_DATALENS_PATH)
    listenings_df["datetime"]  = pd.to_datetime(listenings_df["datetime"])
    listenings_df["date_only"] = pd.to_datetime(listenings_df["date_only"])

    weekday_hour_df = pd.read_csv(USER_WEEKDAY_HOUR_PROFILE_PATH)

    sessions_df = pd.read_csv(LISTENING_SESSIONS_PATH)
    sessions_df["session_start"] = pd.to_datetime(sessions_df["session_start"])
    sessions_df["session_end"]   = pd.to_datetime(sessions_df["session_end"])
    sessions_df["session_date"]  = pd.to_datetime(sessions_df["session_date"])

    user_features_df       = pd.read_csv(USER_FEATURES_PATH)
    user_activity_types_df = pd.read_csv(USER_ACTIVITY_TYPES_PATH)

    return {
        "listenings":          listenings_df,
        "weekday_hour":        weekday_hour_df,
        "sessions":            sessions_df,
        "user_features":       user_features_df,
        "user_activity_types": user_activity_types_df,
    }


def save_figure(filename: str):
    plt.savefig(FIGURES_DIR / filename, bbox_inches="tight", dpi=150)
    plt.close()


def get_axis_limits(series: pd.Series, padding_ratio: float = 0.15, min_padding: float = 0.01) -> tuple[float, float]:
    values = pd.to_numeric(series, errors="coerce").dropna()
    assert not values.empty, (
        "Cannot calculate axis limits for an empty series."
    )

    min_value = float(values.min())
    max_value = float(values.max())

    if min_value == max_value:
        padding = max(abs(min_value) * padding_ratio, min_padding)
    else:
        padding = max((max_value - min_value) * padding_ratio, min_padding)

    return min_value - padding, max_value + padding


def annotate_points(df: pd.DataFrame, x_col: str, y_col: str):
    for _, row in df.iterrows():
        plt.annotate(
            row["username"],
            (row[x_col], row[y_col]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
        )

## 5. Функции построения графиков

In [15]:
def build_plays_by_date(listenings_df: pd.DataFrame):
    plays_by_date = (
        listenings_df.groupby("date_only", as_index=False)["listening_id"]
        .count()
        .rename(columns={"listening_id": "plays_count"})
    )

    plt.figure(figsize=(12, 8))
    plt.plot(plays_by_date["date_only"], plays_by_date["plays_count"], linewidth=2)
    plt.title("Прослушивания по датам")
    plt.xlabel("Дата")
    plt.ylabel("Количество прослушиваний")
    plt.grid(alpha=0.3)

    save_figure("01_plays_by_date.png")


def build_plays_by_hour(listenings_df: pd.DataFrame):
    plays_by_hour = (
        listenings_df.groupby("hour", as_index=False)["listening_id"]
        .count()
        .rename(columns={"listening_id": "plays_count"})
    )

    plt.figure(figsize=(10, 5))
    plt.bar(plays_by_hour["hour"], plays_by_hour["plays_count"], width=0.8)
    plt.title("Прослушивания по часам")
    plt.xlabel("Час")
    plt.ylabel("Количество прослушиваний")
    plt.xticks(range(24))
    plt.xlim(-0.6, 23.6)
    plt.grid(axis="y", alpha=0.3)

    save_figure("02_plays_by_hour.png")


def build_plays_by_weekday(listenings_df: pd.DataFrame):
    plays_by_weekday = (
        listenings_df.groupby(["weekday_number", "weekday"], as_index=False)["listening_id"]
        .count()
        .rename(columns={"listening_id": "plays_count"})
    )
    plays_by_weekday["weekday"] = pd.Categorical(
        plays_by_weekday["weekday"],
        categories=WEEKDAY_ORDER,
        ordered=True,
    )
    plays_by_weekday = plays_by_weekday.sort_values("weekday_number")

    plt.figure(figsize=(10, 5))
    plt.barh(plays_by_weekday["weekday"], plays_by_weekday["plays_count"])
    plt.title("Прослушивания по дням недели")
    plt.xlabel("Количество прослушиваний")
    plt.ylabel("День недели")
    plt.grid(axis="x", alpha=0.3)

    save_figure("03_plays_by_weekday.png")


def build_weekday_hour_heatmap(weekday_hour_df: pd.DataFrame):
    heatmap_df = (
        weekday_hour_df.groupby(["weekday_number", "weekday", "hour"], as_index=False)["plays_count"]
        .sum()
    )
    heatmap_df["weekday"] = pd.Categorical(
        heatmap_df["weekday"],
        categories=WEEKDAY_ORDER,
        ordered=True,
    )
    heatmap_df = heatmap_df.sort_values(["weekday_number", "hour"])

    pivot_df = heatmap_df.pivot_table(
        index="weekday",
        columns="hour",
        values="plays_count",
        aggfunc="sum",
        fill_value=0,
    ).reindex(WEEKDAY_ORDER)

    plt.figure(figsize=(12, 4))
    plt.title("Тепловая карта по дням недели и часам")
    plt.imshow(pivot_df, aspect="auto", cmap="YlOrRd")
    plt.xlabel("Час")
    plt.ylabel("День недели")
    plt.xticks(range(24), range(24))
    plt.yticks(range(len(WEEKDAY_ORDER)), WEEKDAY_ORDER)
    plt.colorbar(label="Количество прослушиваний")

    save_figure("04_weekday_hour_heatmap.png")


def build_day_part_distribution(listenings_df: pd.DataFrame):
    day_part_df = (
        listenings_df.groupby("day_part", as_index=False)["listening_id"]
        .count()
        .rename(columns={"listening_id": "plays_count"})
    )
    day_part_df["day_part"] = pd.Categorical(
        day_part_df["day_part"],
        categories=DAY_PART_ORDER,
        ordered=True,
    )
    day_part_df = day_part_df.sort_values("day_part")

    plt.figure(figsize=(8, 8))
    plt.title("Прослушивания по частям суток")
    plt.bar(day_part_df["day_part"], day_part_df["plays_count"])
    plt.xlabel("Часть суток")
    plt.ylabel("Количество прослушиваний")
    plt.grid(axis="y", alpha=0.3)

    save_figure("05_day_part_distribution.png")


def build_session_duration_distribution(sessions_df: pd.DataFrame):
    plt.figure(figsize=(10, 5))
    plt.title("Распределение длительности сессий (логарифмическая шкала)")
    plt.hist(sessions_df["duration_minutes"], bins=30, log=True)
    plt.xlabel("Длительность, минут")
    plt.ylabel("Количество сессий")
    plt.grid(axis="y", alpha=0.3)

    save_figure("06_session_duration_distribution.png")


def build_user_regularity_intensity_scatter(user_features_df: pd.DataFrame):
    x_limits = get_axis_limits(user_features_df["active_days_share"], padding_ratio=0.2)
    y_limits = get_axis_limits(user_features_df["avg_daily_plays"], padding_ratio=0.1)

    plt.figure(figsize=(9, 6))
    plt.scatter(
        user_features_df["active_days_share"],
        user_features_df["avg_daily_plays"],
        s=80,
        alpha=0.8,
    )
    plt.title("Регулярность и интенсивность пользователей")
    annotate_points(user_features_df, "active_days_share", "avg_daily_plays")
    plt.xlim(x_limits)
    plt.ylim(y_limits)
    plt.xlabel("Доля активных дней")
    plt.ylabel("Среднее число прослушиваний в активный день")
    plt.grid(alpha=0.3)

    save_figure("07_user_regularity_intensity_scatter.png")


def build_user_night_weekend_scatter(user_features_df: pd.DataFrame):
    x_limits = get_axis_limits(user_features_df["night_share"], padding_ratio=0.2)
    y_limits = get_axis_limits(user_features_df["weekend_share"], padding_ratio=0.1)

    plt.figure(figsize=(9, 6))
    plt.scatter(
        user_features_df["night_share"],
        user_features_df["weekend_share"],
        s=80,
        alpha=0.8,
    )
    plt.title("Доля прослушиваний ночью и в выходные")
    annotate_points(user_features_df, "night_share", "weekend_share")
    plt.xlim(x_limits)
    plt.ylim(y_limits)
    plt.xlabel("Доля прослушиваний ночью")
    plt.ylabel("Доля прослушиваний в выходные")
    plt.grid(alpha=0.3)

    save_figure("08_user_night_weekend_scatter.png")


def build_listener_types_distribution(user_activity_types_df: pd.DataFrame):
    listener_type_df = (
        user_activity_types_df.groupby("listener_type", as_index=False)["username"]
        .count()
        .rename(columns={"username": "users_count"})
        .sort_values("users_count", ascending=False)
    )

    plt.figure(figsize=(6, 3))
    plt.title("Распределение типов слушателей")
    plt.barh(listener_type_df["listener_type"], listener_type_df["users_count"])
    plt.xlabel("Количество пользователей")
    plt.ylabel("Тип слушателя")
    plt.grid(axis="x", alpha=0.3)

    save_figure("09_listener_types_distribution.png")

## 6. Функции построения summary-таблиц и финальных проверок

In [16]:
def build_dataset_summary(
    listenings_df: pd.DataFrame,
    sessions_df: pd.DataFrame,
    user_features_df: pd.DataFrame,
):
    dataset_summary_df = pd.DataFrame(
        {
            "metric": [
                "total_plays",
                "unique_users",
                "period_start",
                "period_end",
                "total_sessions",
                "avg_plays_per_user",
                "avg_active_days_per_user",
            ],
            "value": [
                int(len(listenings_df)),
                int(listenings_df["username"].nunique()),
                listenings_df["date_only"].min().date().isoformat(),
                listenings_df["date_only"].max().date().isoformat(),
                int(len(sessions_df)),
                round(user_features_df["total_plays"].mean(), 6),
                round(user_features_df["active_days"].mean(), 6),
            ],
        }
    )
    dataset_summary_df.to_csv(SUMMARY_DIR / "dataset_summary.csv", index=False)


def build_user_features_summary(user_features_df: pd.DataFrame):
    summary_columns = [
        "total_plays",
        "active_days",
        "active_days_share",
        "avg_daily_plays",
        "median_daily_plays",
        "max_daily_plays",
        "std_daily_plays",
        "night_share",
        "evening_share",
        "weekend_share",
        "session_count",
        "avg_session_length",
    ]
    user_features_summary_df = (
        user_features_df[summary_columns]
        .describe()
        .transpose()
        .reset_index()
        .rename(columns={"index": "feature"})
    )
    user_features_summary_df.to_csv(SUMMARY_DIR / "user_features_summary.csv", index=False)


def build_listener_type_summary(user_activity_types_df: pd.DataFrame):
    listener_type_summary_df = (
        user_activity_types_df.groupby("listener_type", as_index=False)
        .agg(
            users_count          =("username", "count"),
            avg_total_plays      =("total_plays", "mean"),
            avg_active_days      =("active_days", "mean"),
            avg_active_days_share=("active_days_share", "mean"),
            avg_daily_plays      =("avg_daily_plays", "mean"),
            avg_evening_share    =("evening_share", "mean"),
            avg_night_share      =("night_share", "mean"),
            avg_weekend_share    =("weekend_share", "mean"),
        )
        .sort_values("users_count", ascending=False)
    )
    numeric_columns = listener_type_summary_df.select_dtypes(include="number").columns
    listener_type_summary_df[numeric_columns] = listener_type_summary_df[numeric_columns].round(6)
    listener_type_summary_df.to_csv(SUMMARY_DIR / "listener_type_summary.csv", index=False)


def check_columns(df: pd.DataFrame, required_columns: list[str], dataframe_name: str):
    missing_columns = sorted(set(required_columns) - set(df.columns))
    assert len(missing_columns) == 0, f"Missing columns in {dataframe_name}: {missing_columns}"


def check_share_sum(series: pd.Series, target: float, dataframe_name: str, label: str):
    max_deviation = (series - target).abs().max()
    assert pd.notna(max_deviation), (
        f"Share check failed in {dataframe_name} for {label}. Max deviation is NaN."
    )
    assert max_deviation <= SHARE_TOLERANCE, (
        f"Share check failed in {dataframe_name} for {label}. Max deviation: {max_deviation}"
    )


def run_checks(listenings_df: pd.DataFrame, user_features_df: pd.DataFrame, user_activity_types_df: pd.DataFrame):
    assert not listenings_df.empty, "listenings_datalens.csv is empty."

    check_columns(user_features_df, ["username"], "user_features.csv")
    check_columns(
        user_activity_types_df,
        ["username", "listener_type", "behavior_profile"],
        "user_activity_types.csv",
    )

    assert user_features_df["username"].nunique() == EXPECTED_USER_COUNT
    assert user_activity_types_df["username"].nunique() == EXPECTED_USER_COUNT

    assert not user_features_df["username"].isna().any()
    assert not user_activity_types_df["username"].isna().any()
    assert not user_activity_types_df["listener_type"].isna().any()

    time_share_sum = (
        user_features_df["morning_share"]
        + user_features_df["day_share"]
        + user_features_df["evening_share"]
        + user_features_df["night_share"]
    )
    check_share_sum(time_share_sum, 1.0, "user_features.csv", "day part shares")

    week_share_sum = user_features_df["weekday_share"] + user_features_df["weekend_share"]
    check_share_sum(week_share_sum, 1.0, "user_features.csv", "weekday/weekend shares")

## 7. Загрузка итоговых таблиц

In [17]:
data = load_data()

listenings_df          = data["listenings"]
weekday_hour_df        = data["weekday_hour"]
sessions_df            = data["sessions"]
user_features_df       = data["user_features"]
user_activity_types_df = data["user_activity_types"]

print(f"listenings_df:          {listenings_df.shape}")
print(f"weekday_hour_df:        {weekday_hour_df.shape}")
print(f"sessions_df:            {sessions_df.shape}")
print(f"user_features_df:       {user_features_df.shape}")
print(f"user_activity_types_df: {user_activity_types_df.shape}")

listenings_df:          (166153, 13)
weekday_hour_df:        (1848, 9)
sessions_df:            (1465, 19)
user_features_df:       (11, 27)
user_activity_types_df: (11, 18)


## 8. Построение графиков, summary-таблиц и запуск финальных проверок

In [18]:
build_plays_by_date(listenings_df)
build_plays_by_hour(listenings_df)
build_plays_by_weekday(listenings_df)
build_weekday_hour_heatmap(weekday_hour_df)
build_day_part_distribution(listenings_df)
build_session_duration_distribution(sessions_df)
build_user_regularity_intensity_scatter(user_features_df)
build_user_night_weekend_scatter(user_features_df)
build_listener_types_distribution(user_activity_types_df)

build_dataset_summary(listenings_df, sessions_df, user_features_df)
build_user_features_summary(user_features_df)
build_listener_type_summary(user_activity_types_df)

run_checks(listenings_df, user_features_df, user_activity_types_df)
print("Report artifacts built successfully.")

Report artifacts built successfully.


## 9. Список сохранённых графиков

In [19]:
sorted(path.name for path in FIGURES_DIR.glob("*.png"))

['01_plays_by_date.png',
 '02_plays_by_hour.png',
 '03_plays_by_weekday.png',
 '04_weekday_hour_heatmap.png',
 '05_day_part_distribution.png',
 '06_session_duration_distribution.png',
 '07_user_regularity_intensity_scatter.png',
 '08_user_night_weekend_scatter.png',
 '09_listener_types_distribution.png']

## 10. Список сохранённых summary-таблиц

In [20]:
sorted(path.name for path in SUMMARY_DIR.glob("*.csv"))

['dataset_summary.csv',
 'listener_type_summary.csv',
 'user_features_summary.csv']